extra code for help

In [ ]:
# Combine two overlapping Harvest exports into one, keeping the newer rows and saving an audit of the overlaps


import pandas as pd
input_dir = "../data/extracts/"

old_file = input_dir + "harvest_kwh_15min_250723-260508.csv"
new_file = input_dir + "harvest_kwh_15min_260504-260713.csv"

output_file = "harvest_kwh_15min_250723-260713.csv"
audit_file = "harvest_kwh_overlap_audit.csv"

# Read both exports
old = pd.read_csv(old_file)
new = pd.read_csv(new_file)

# Clean column names
old.columns = old.columns.str.strip()
new.columns = new.columns.str.strip()

# Make sure the files have the same columns
if set(old.columns) != set(new.columns):
    raise ValueError(
        "The columns do not match.\n"
        f"Only in older file: {sorted(set(old.columns) - set(new.columns))}\n"
        f"Only in newer file: {sorted(set(new.columns) - set(old.columns))}"
    )

# Put newer file columns in the same order as the older file
new = new[old.columns]

# Parse timestamps
old["datetime"] = pd.to_datetime(old["datetime"], errors="coerce")
new["datetime"] = pd.to_datetime(new["datetime"], errors="coerce")

if old["datetime"].isna().any():
    raise ValueError(
        f"Older file contains {old['datetime'].isna().sum()} invalid timestamps."
    )

if new["datetime"].isna().any():
    raise ValueError(
        f"Newer file contains {new['datetime'].isna().sum()} invalid timestamps."
    )

# Long-format Harvest data: one unique row per meter and timestamp
if "meter_name" in old.columns:
    old["meter_name"] = old["meter_name"].astype("string").str.strip()
    new["meter_name"] = new["meter_name"].astype("string").str.strip()
    duplicate_key = ["datetime", "meter_name"]
else:
    # Use this only if these are wide-format files
    duplicate_key = ["datetime"]

# Record the source so overlaps can be audited
old["_source_file"] = old_file
new["_source_file"] = new_file

# Older first, newer second
all_rows = pd.concat([old, new], ignore_index=True)

# Save all overlapping rows for review
duplicate_mask = all_rows.duplicated(
    subset=duplicate_key,
    keep=False
)

overlap_audit = (
    all_rows.loc[duplicate_mask]
    .sort_values(duplicate_key + ["_source_file"])
)

overlap_audit.to_csv(
    audit_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

# Keep the last duplicate, so the newer export wins
combined = (
    all_rows
    .drop_duplicates(subset=duplicate_key, keep="last")
    .drop(columns="_source_file")
    .sort_values(duplicate_key)
    .reset_index(drop=True)
)

# Final validation
assert not combined.duplicated(subset=duplicate_key).any()

combined.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print(f"Older rows:           {len(old):,}")
print(f"Newer rows:           {len(new):,}")
print(f"Rows before dedupe:   {len(all_rows):,}")
print(f"Rows removed:         {len(all_rows) - len(combined):,}")
print(f"Combined rows:        {len(combined):,}")
print(f"First timestamp:      {combined['datetime'].min()}")
print(f"Last timestamp:       {combined['datetime'].max()}")
print(f"Combined file saved:  {output_file}")
print(f"Overlap audit saved:  {audit_file}")

In [ ]:
# finding min and max datetime in the data file

import pandas as pd
input_dir = "../data/extracts/"

var_file = input_dir + 'fy26_aurora_data.csv' #'harvest_kwh_15min_250723-260713.csv'

df = pd.read_csv(var_file, usecols=["datetime"])
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

min_dt = df["datetime"].min()
max_dt = df["datetime"].max()

print("min datetime:", min_dt)
print("max datetime:", max_dt)

In [ ]:
# Fill harvests missing start-of-year data using Aurora
import pandas as pd

input_dir = "../data/extracts/"

harvest_file = input_dir + "harvest_kwh_15min_250723-260713.csv"
aurora_file = input_dir + "fy26_aurora_data.csv"

fy_start = pd.Timestamp("2025-07-01 00:00:00")

# read Harvest (long format: datetime, meter_name, meter_reading)
harvest = pd.read_csv(harvest_file)
harvest["datetime"] = pd.to_datetime(harvest["datetime"])
harvest["meter_name"] = harvest["meter_name"].astype("string").str.strip()

# read Aurora (long format: datetime, meter_reading, meter_name, stuck)
aurora = pd.read_csv(aurora_file)
aurora["datetime"] = pd.to_datetime(aurora["datetime"])
aurora["meter_name"] = aurora["meter_name"].astype("string").str.strip()

harvest_start = harvest["datetime"].min()
print(f"Harvest data starts at: {harvest_start}")
print(f"Filling gap from {fy_start} through {harvest_start} using Aurora data")

# only backfill meters that actually exist in Harvest, and only rows
# strictly before Harvest's earliest timestamp (no overlap/duplication)
harvest_meters = set(harvest["meter_name"].unique())

gap_mask = (
    (aurora["datetime"] >= fy_start)
    & (aurora["datetime"] < harvest_start)
    & (aurora["meter_name"].isin(harvest_meters))
)
aurora_gap = aurora.loc[gap_mask].copy()

# drop Aurora's flagged "stuck" readings - leave those as missing timestamps
# so the notebook's normal special-meter/interpolation logic handles them
stuck_dropped = int((aurora_gap["stuck"] == 1).sum())
aurora_gap = aurora_gap.loc[aurora_gap["stuck"] == 0, ["datetime", "meter_name", "meter_reading"]]

# report meters where Aurora had no usable (non-stuck) data anywhere in the gap window
meters_no_gap_data = sorted(harvest_meters - set(aurora_gap["meter_name"].unique()))

combined = (
    pd.concat([aurora_gap, harvest], ignore_index=True)
    .sort_values(["meter_name", "datetime"])
    .reset_index(drop=True)
)

assert not combined.duplicated(subset=["datetime", "meter_name"]).any()

min_date = combined['datetime'].min().strftime('%y%m%d')
max_date = combined['datetime'].max().strftime('%y%m%d')

output_file = input_dir + f"harvest_kwh_15min_{min_date}-{max_date}.csv"

combined.to_csv(output_file, index=False, date_format="%Y-%m-%d %H:%M:%S")

print(f"Aurora rows added:         {len(aurora_gap):,}")
print(f"Aurora stuck rows dropped: {stuck_dropped:,}")
print(f"Harvest rows (unchanged):  {len(harvest):,}")
print(f"Combined rows:             {len(combined):,}")
print(f"First timestamp:           {combined['datetime'].min()}")
print(f"Last timestamp:            {combined['datetime'].max()}")
print(f"Combined file saved:       {output_file}")

if meters_no_gap_data:
    print(f"\n{len(meters_no_gap_data)} meter(s) had no usable Aurora data in the gap window:")
    for name in meters_no_gap_data:
        print(f"  - {name}")

Harvest data starts at: 2025-07-23 09:45:00
Filling gap from 2025-07-01 00:00:00 through 2025-07-23 09:45:00 using Aurora data
Aurora rows added:         178,583
Aurora stuck rows dropped: 23,612
Harvest rows (unchanged):  2,049,522
Combined rows:             2,228,105
First timestamp:           2025-07-01 00:00:00
Last timestamp:            2026-07-13 10:45:00
Combined file saved:       ../data/extracts/harvest_kwh_15min_250701-260713.csv
